# Latent Depth Controller Loss Analysis

Analyze binary STOP/CONTINUE controller training runs produced by `run_scripts/run_latent_depth_controller_training.sh`.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'lvar').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_ROOT = PROJECT_ROOT / 'outputs' / 'latent_depth_controller_m3cot'
MANIFEST_PATH = RUN_ROOT / 'latent_depth_controller_runs.jsonl'
print('RUN_ROOT =', RUN_ROOT)
print('MANIFEST_PATH =', MANIFEST_PATH)

## Load Runs

In [ ]:
def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

manifest_rows = read_jsonl(MANIFEST_PATH) if MANIFEST_PATH.exists() else []
if not manifest_rows and RUN_ROOT.exists():
    for summary_path in RUN_ROOT.glob('**/latent_depth_controller_summary.json'):
        summary = read_json(summary_path)
        manifest_rows.append({
            'output_dir': str(summary_path.parent),
            'summary_path': str(summary_path),
            'loss_history_path': summary.get('loss_history_path', str(summary_path.parent / 'latent_depth_controller_losses.jsonl')),
            'checkpoint_path': summary.get('checkpoint_path'),
            'dataset_partition': summary.get('dataset_partition'),
            'target_policy': summary.get('target_policy'),
            'context': summary.get('context'),
        })

runs = []
for row in manifest_rows:
    summary_path = Path(row.get('summary_path') or Path(row['output_dir']) / 'latent_depth_controller_summary.json')
    summary = read_json(summary_path) if summary_path.exists() else {}
    train_metrics = summary.get('train_metrics') or {}
    val_metrics = summary.get('validation_metrics') or {}
    runs.append({
        **row,
        'summary_path': str(summary_path),
        'target_policy': summary.get('target_policy', row.get('target_policy')),
        'dataset_partition': summary.get('dataset_partition', row.get('dataset_partition')),
        'context': summary.get('context', row.get('context')),
        'num_train_rows': summary.get('num_train_rows'),
        'num_validation_rows': summary.get('num_validation_rows'),
        'train_loss': train_metrics.get('loss'),
        'train_binary_accuracy': train_metrics.get('binary_accuracy'),
        'validation_loss': val_metrics.get('loss'),
        'validation_binary_accuracy': val_metrics.get('binary_accuracy'),
    })

runs_df = pd.DataFrame(runs)
display(runs_df if not runs_df.empty else runs_df)

## Loss Curves

In [ ]:
loss_rows = []
for run_idx, run in runs_df.reset_index(drop=True).iterrows() if not runs_df.empty else []:
    loss_path = Path(run.get('loss_history_path') or Path(run['output_dir']) / 'latent_depth_controller_losses.jsonl')
    if not loss_path.exists():
        print('Missing loss history:', loss_path)
        continue
    for row in read_jsonl(loss_path):
        loss_rows.append({
            **row,
            'run_idx': run_idx,
            'run_label': f"{run.get('dataset_partition', '')}/{run.get('target_policy', '')}/{Path(run.get('output_dir', '')).name}",
            'output_dir': run.get('output_dir'),
        })

loss_df = pd.DataFrame(loss_rows)
print(f'Loaded {len(loss_df):,} loss rows')
display(loss_df.head() if not loss_df.empty else loss_df)

In [ ]:
if loss_df.empty:
    print('No loss rows available yet.')
else:
    plt.figure(figsize=(10, 5))
    for label, group in loss_df.groupby('run_label'):
        smooth = group.sort_values('global_step')['loss'].rolling(25, min_periods=1).mean()
        plt.plot(group.sort_values('global_step')['global_step'], smooth, label=label)
    plt.xlabel('Global step')
    plt.ylabel('BCE loss')
    plt.title('Latent-depth controller training loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

## Depth Targets And Run Metrics

In [ ]:
if not loss_df.empty:
    depth_counts = loss_df.groupby(['run_label', 'earliest_correct_depth'], as_index=False).size()
    display(depth_counts)
    pivot = depth_counts.pivot_table(index='earliest_correct_depth', columns='run_label', values='size', fill_value=0)
    ax = pivot.plot(kind='bar', figsize=(10, 4))
    ax.set_xlabel('Earliest correct latent depth')
    ax.set_ylabel('Training examples')
    ax.set_title('Target depth distribution in training logs')
    plt.show()

metric_cols = ['dataset_partition', 'target_policy', 'context', 'num_train_rows', 'num_validation_rows', 'train_loss', 'train_binary_accuracy', 'validation_loss', 'validation_binary_accuracy']
if not runs_df.empty:
    display(runs_df[[col for col in metric_cols if col in runs_df.columns]])

## Predicted vs Target Depth Distributions

In [ ]:
dist_rows = []
for _, run in runs_df.iterrows() if not runs_df.empty else []:
    summary_path = Path(run['summary_path'])
    if not summary_path.exists():
        continue
    summary = read_json(summary_path)
    label = f"{run.get('dataset_partition', '')}/{run.get('target_policy', '')}/{Path(run.get('output_dir', '')).name}"
    for split_name, metrics in [('train', summary.get('train_metrics') or {}), ('validation', summary.get('validation_metrics') or {})]:
        for kind, values in [
            ('predicted', metrics.get('predicted_depth_distribution') or {}),
            ('target', metrics.get('target_earliest_depth_distribution') or {}),
        ]:
            for depth, count in values.items():
                dist_rows.append({'run_label': label, 'split': split_name, 'kind': kind, 'depth': int(depth), 'count': int(count)})

dist_df = pd.DataFrame(dist_rows)
display(dist_df.head() if not dist_df.empty else dist_df)
if not dist_df.empty:
    for (label, split_name), group in dist_df.groupby(['run_label', 'split']):
        pivot = group.pivot_table(index='depth', columns='kind', values='count', fill_value=0)
        ax = pivot.sort_index().plot(kind='bar', figsize=(8, 3))
        ax.set_title(f'{label} - {split_name} depth distribution')
        ax.set_xlabel('Depth')
        ax.set_ylabel('Examples')
        plt.show()